# Generate Synthetic Historical RFQs

This notebook generates a synthetic historical RFQ dataset used to test RFQ conversion prediction.

Clients differ in activity, typical trade size and trading behaviour. Market conditions and product characteristics are also simulated.

All data is synthetic and generated for research and learning purposes.

## 1. Generate RFQs

Generate the main RFQ characteristics, including client, product, underlying and direction.

In [ ]:
import pandas as pd 
import numpy as np 

# random number generator
rng = np.random.default_rng(100)

N_RFQS = 5_000
N_CLIENTS = 40

rfq_id = range(1,N_RFQS+1)

clients = [f"Client{i:02d}" for i in range(1, N_CLIENTS + 1)]

product_types = ["Autocall", "Barrier Reverse Convertible", "Reverse Convertible", "Bonus Certificate", "Tracker"]

underlyings = ["NVDA", "AMZN", "TSLA", "AAPL", "MSFT", "NESN", "SPX", "SMI",]

# Synthetic product mix, with Autocalls intentionally dominant
product_probabilities = [0.64, 0.12, 0.08, 0.09, 0.07,]

# Generate product type for each synthetic RFQ
rfq_product_type = rng.choice(product_types, size = N_RFQS, p = product_probabilities)

# --- Clients

# generate different activity levels
client_activity_weights = rng.lognormal(mean = 0, sigma = 0.4, size = N_CLIENTS)

#convert activity weights into probabilities
client_probabilities = (client_activity_weights/client_activity_weights.sum())

# assign a client to each rfq
rfq_clients = rng.choice(clients, size=N_RFQS, p=client_probabilities)

# generate one underyling for every synthetic rfq (equally distributed)
rfq_underlyings= rng.choice(underlyings, size=N_RFQS)


# Generate one direction for each synthetic RFQ
rfq_directions = rng.choice(["BUY", "SELL"], size=N_RFQS)

# Create historical RFQ Dataset

historical_rfqs = pd.DataFrame({
    "rfq_id": rfq_id,
    "client":rfq_clients, 
    "product_type": rfq_product_type, 
    "underlying": rfq_underlyings,
    "direction":rfq_directions
    })


In [2]:
# --- Notional

# every client has different typical trade size
client_typical_notionals = dict(
    zip(
        clients,
        rng.lognormal(mean=np.log(500_000), sigma=0.5, size=N_CLIENTS)
    )
)
typical_notional = historical_rfqs["client"].map(client_typical_notionals)

notionals = rng.lognormal(mean=np.log(typical_notional),sigma=0.5)

historical_rfqs["notional"]= (np.clip(notionals,50_000,5_000_000)/10_000).round()*10_000


# --- Market conditions at time of RFQ
historical_rfqs["underlying_move_pct"] = np.clip( 
    rng.normal(0, 2, N_RFQS), -10, 10
)

historical_rfqs["iv_change"] = np.clip(
    rng.normal(0, 3, N_RFQS), -12, 12
)

historical_rfqs["bid_ask_spread_pct"] = np.clip(
    rng.lognormal(mean=np.log(0.4), sigma=0.5, size=N_RFQS), 0.05, 2
)


# --- Distance to barrier

barrier_products = ["Autocall", "Barrier Reverse Convertible", "Bonus Certificate" ]

has_barrier = historical_rfqs["product_type"].isin(barrier_products)

historical_rfqs["distance_to_barrier_pct"] = np.where(
    has_barrier,
    rng.uniform(0, 50, N_RFQS),
    np.nan
)

## 2. Generate Trade Outcomes

Trade probabilities are simulated using client behaviour, product preferences, notional size and market conditions.

In [3]:
# --- client behaviour 

client_trade_propensity = dict( 
    zip(
        clients,
        rng.normal(0, 0.6, N_CLIENTS)
    )
)

client_preferred_product = dict (
    zip(
        clients,
        rng.choice(product_types, size=N_CLIENTS, p=product_probabilities)
    )
)

# Probability that RFQ converts into a trade
client_effect = historical_rfqs["client"].map(client_trade_propensity)


preferred_product = (
    historical_rfqs["product_type"]  == historical_rfqs["client"].map(client_preferred_product)
).astype(int)

abs_market_move = historical_rfqs["underlying_move_pct"].abs()

abs_iv_move = historical_rfqs["iv_change"].abs()

notional_mn = historical_rfqs["notional"] / 1_000_000


# Synthetic log-odds of trading
trade_logit = (
    -0.5
    + client_effect
    + 0.50 * preferred_product
    - 0.20 * notional_mn
    - 0.70 * historical_rfqs["bid_ask_spread_pct"]
    + 0.08 * abs_market_move 
    + 0.04 * abs_iv_move
)

# Convert log-odds into probability between 0 and 1
trade_probability = 1 / (1 + np.exp(-trade_logit))


# Generate the final target: 1 = traded, 0 = did not trade
historical_rfqs["traded"] = rng.binomial( 1, trade_probability)


## 3. Create Historical Dataset

Assign timestamps, sort the RFQs chronologically and save the final dataset.

In [ ]:
# --- Timestamp

# Generate business days only (Monday to Friday)
business_days = pd.bdate_range(
    start="2025-01-01",
    end="2025-01-31"
)

# Randomly assign one business day to each RFQ
random_days = rng.choice(business_days, size=N_RFQS)

# Generate a random time during desk hours
random_minutes = rng.integers( 8 * 60, 18 * 60, size=N_RFQS)

# Combine day and time
historical_rfqs["timestamp"] = (
    pd.to_datetime(random_days)
    + pd.to_timedelta(random_minutes, unit="m")
)

# Sort RFQs chronologically
historical_rfqs = historical_rfqs.sort_values("timestamp").reset_index(drop=True)

# Save synthetic historical RFQ dataset
historical_rfqs.to_csv(
    "../data/synthetic_historical_rfqs.csv",
    index=False
)

## 4. Sanity Checks

Check that the generated dataset behaves consistently with the assumptions used to simulate trade conversion.

In [5]:
historical_rfqs.head()

,rfq_id,client,product_type,underlying,direction,notional,underlying_move_pct,iv_change,bid_ask_spread_pct,distance_to_barrier_pct,traded,timestamp
0,3130,Client32,Autocall,NVDA,SELL,1080000.0,2.031788,-0.594538,0.347405,1.955365,1,2025-01-01 08:03:00
1,4218,Client34,Autocall,NESN,BUY,380000.0,1.700107,3.292242,0.629700,11.808964,0,2025-01-01 08:03:00
2,1834,Client31,Autocall,TSLA,BUY,800000.0,1.455117,2.606704,0.338126,39.432096,1,2025-01-01 08:07:00
3,3278,Client18,Reverse Convertible,AAPL,BUY,850000.0,5.295314,3.887962,0.180865,NaN,1,2025-01-01 08:09:00
4,1105,Client37,Autocall,AAPL,SELL,290000.0,-2.833395,-0.971015,0.638075,25.495218,0,2025-01-01 08:11:00


In [6]:
# Overall trade rate
historical_rfqs["traded"].mean()

np.float64(0.4202)

In [7]:
# Trade rate by product type
historical_rfqs.groupby("product_type")["traded"].mean().sort_values(ascending=False)

product_type
Autocall                       0.442947
Barrier Reverse Convertible    0.414944
Tracker                        0.371875
Reverse Convertible            0.359801
Bonus Certificate              0.355895
Name: traded, dtype: float64

In [8]:
sanity_check = (
    historical_rfqs
    .assign(
        abs_underlying_move_pct=historical_rfqs["underlying_move_pct"].abs(),
        abs_iv_change=historical_rfqs["iv_change"].abs()
    )
    .groupby("traded")[[
        "notional",
        "bid_ask_spread_pct",
        "abs_underlying_move_pct",
        "abs_iv_change"
    ]]
    .mean()
    .round(3)
)

sanity_check

,notional,bid_ask_spread_pct,abs_underlying_move_pct,abs_iv_change
traded,,,,
0,687164.539,0.468,1.540,2.365
1,659852.451,0.430,1.634,2.507
